# Reusing microscopy images with IIIF
**4TU.ResearchData · Hands-on workshop · Google Colab**

**Research question:** How can we inspect, annotate and computationally highlight cells in microscopy images without downloading the entire dataset?

We will discover a dataset, read its IIIF manifest, retrieve an image through the Image API and highlight candidate cells using Python.

Run the code cells in order. Change the parameters when prompted and observe how the results change.

## 0. Set up the notebook
Google Colab provides many scientific Python libraries already. Run this cell to ensure the required packages are available. No local environment is needed.

In [ ]:
%pip install -q requests pillow numpy matplotlib scikit-image

In [ ]:
# HTTP requests and JSON inspection
import requests
import json
from uuid import UUID

# Reading and displaying images
from io import BytesIO
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Image processing and measurement
from skimage import color, filters, morphology, measure

## 1. Discover a reusable dataset
Open the [sample dataset in the 4TU.ResearchData sandbox](https://next.data.4tu.nl/datasets/5d40e250-23bd-4a79-9626-6c9d6003c747) (short link: https://edu.nl/3bdn4).

Read the description, methods, licence and file list. What kind of microscopy is used? What information would you need to distinguish living and dead cells?

## 2. Retrieve and explore the IIIF manifest
The Presentation API describes what a viewer should display. The manifest is a JSON-LD document containing links, metadata and a list of Canvases.

A **Canvas** is a coordinate space. A painting annotation links an image to that space. An Image API service provides image representations and zoom tiles. Canvas dimensions are not necessarily the same as source image dimensions.

In [ ]:
# Define the repository and dataset identifier.
base = "https://next.data.4tu.nl"
dataset_uuid = "5d40e250-23bd-4a79-9626-6c9d6003c747"

# Build the Presentation API endpoint.
manifest_url = f"{base}/iiif/v3/{dataset_uuid}/1/manifest"
print(manifest_url)

# Request the manifest and check whether the request succeeded.
response = requests.get(manifest_url, timeout=60)
response.raise_for_status()

# Convert the JSON response into a Python dictionary.
original = response.json()
print(list(original))

In [ ]:
# Inspect the human-readable label, metadata and number of Canvases.
print("Title:", original.get("label"))
print("Metadata:", original.get("metadata"))
print("Number of Canvases:", len(original["items"]))

# Select and inspect the first Canvas (Python indexing starts at zero).
canvas = original["items"][0]
print(json.dumps(canvas, indent=2))

## 3. Explore and annotate in a viewer
1. Open the [Digirati Manifest Editor](https://manifest-editor.digirati.services/) and load the manifest URL printed above.
2. Compare the title, metadata and Canvas list with the JSON. Inspect the raw manifest in the preview menu if available.
3. Select an image and add an annotation page and a spatial annotation. Draw a circle or polygon around a candidate cell and describe it. Only use a viability label if the experimental evidence supports it.
4. Preview the result in Theseus and enable annotations. Export the edited manifest as JSON.

**Discuss:** Which parts of the JSON have changed? Have the original image pixels changed? A derived manifest can be published separately, for example using GitHub Pages, while preserving links to the source images.

## 4. Discover image file UUIDs with the repository API
The repository API returns file metadata; the IIIF Image API returns image representations. These are different APIs.

Production patterns:
- Repository files: `https://data.4tu.nl/v2/articles/<dataset_uuid>/files`
- Image service: `https://data.4tu.nl/iiif/v3/<file_uuid>`

For this exercise, both use the sandbox domain (`next.data.4tu.nl`). Do not assume a sandbox UUID exists in production.

See the [4TU.ResearchData API documentation](https://data.4tu.nl/api/docs?urls.primaryName=v2).

In [ ]:
# Construct the repository API URL for the dataset's files.
files_url = f"{base}/v2/articles/{dataset_uuid}/files"
print(files_url)

# Retrieve file metadata, not the image files themselves.
response = requests.get(files_url, timeout=60)
response.raise_for_status()
file_list = response.json()

# Inspect one complete record to understand the response structure.
print(json.dumps(file_list[0], indent=2))

In [ ]:
# List the first ten files with their index, name and UUID.
# Change the slice if you want to inspect more files.
for i, item in enumerate(file_list[:10]):
    print(i, item.get("name"), item.get("uuid"))

### Build the IIIF Image API endpoint
Choose the index of a microscopy image from the list. The Image API's `info.json` describes image dimensions and supported request features.

In [ ]:
# Change this index to select a microscopy image from the list above.
image_index = 0
file_record = file_list[image_index]

# Extract the identifier and verify that it is a UUID.
file_uuid = file_record.get("uuid") or file_record.get("id")
file_uuid = str(UUID(str(file_uuid)))
print("Selected file:", file_record.get("name"))

# Build the Image API service URL using the FILE UUID, not the dataset UUID.
iiif_base = f"{base}/iiif/v3/{file_uuid}"
print(iiif_base)

In [ ]:
# Ask the image server what it can deliver.
response = requests.get(f"{iiif_base}/info.json", timeout=60)
response.raise_for_status()
info = response.json()
print(json.dumps(info, indent=2))

## 5. Retrieve an image with Python
A IIIF Image API request has the structure:

`{service}/{region}/{size}/{rotation}/{quality}.{format}`

- `full`: the entire image area.
- `800,`: request a width of 800 pixels and preserve the aspect ratio.
- `0`: no rotation.
- `default.jpg`: default quality, JPEG output.

Check `info.json` for supported features and sizes. We start with a bounded preview rather than downloading the original file. If the server does not support arbitrary resizing, select a width and height explicitly advertised in `sizes`.

In [ ]:
# Define each component of the IIIF request separately.
region = "full"
size = f"{min(800, info['width'])},"
rotation = "0"
quality_format = "default.jpg"

# Assemble the request URL.
image_url = f"{iiif_base}/{region}/{size}/{rotation}/{quality_format}"
print(image_url)

In [ ]:
# Request the rendered image and check the HTTP status.
response = requests.get(image_url, timeout=60)
response.raise_for_status()

# Decode the returned bytes as an RGB image (red, green, blue).
image = Image.open(BytesIO(response.content)).convert("RGB")
print("Returned size (width, height):", image.size)

# Display the image in the notebook.
plt.imshow(image)
plt.axis("off")
plt.show()

### Optional: retrieve a region of interest
If `info.json` advertises `regionByPx` or otherwise supports pixel regions, we can request a smaller area using `x,y,width,height` in **source-image pixels**. This is not the same as coordinates in the resized preview. The following example selects a central rectangle. Skip it if pixel cropping is unsupported.

In [ ]:
# Select the central half of the source image in each dimension.
width, height = info["width"], info["height"]
x, y = width // 4, height // 4
crop_width, crop_height = max(1, width // 2), max(1, height // 2)
region = f"{x},{y},{crop_width},{crop_height}"
size = f"{min(800, crop_width)},"

# Request only the selected region.
roi_url = f"{iiif_base}/{region}/{size}/0/default.jpg"
print(roi_url)
response = requests.get(roi_url, timeout=60)
response.raise_for_status()
roi = Image.open(BytesIO(response.content)).convert("RGB")

plt.imshow(roi)
plt.axis("off")
plt.show()

## 6. Highlight candidate cells with Python
We will use intensity thresholding to separate candidate foreground objects from the background, clean the mask and identify connected regions.

This is exploratory segmentation, **not a validated cell viability assay**. A rendered JPEG may not preserve the original fluorescence channels or quantitative intensities. Inspect the experimental description before interpreting detections biologically.

### 6.1 Convert to grayscale and inspect intensities

In [ ]:
# Choose the full-image preview for analysis.
# To analyse the optional crop instead, change image to roi below.
analysis_image = np.asarray(image)

# Convert RGB pixels to a single grayscale intensity per pixel.
gray = color.rgb2gray(analysis_image)
print("Array shape:", gray.shape)

plt.imshow(gray, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
# Plot the distribution of grayscale intensities (0 = dark, 1 = bright).
# ravel() turns the 2D image into a 1D array of pixels.
plt.hist(gray.ravel(), bins=50)
plt.xlabel("Intensity (0–1)")
plt.ylabel("Pixel count")
plt.show()

### 6.2 Smooth and threshold the image
Gaussian smoothing reduces small intensity fluctuations. Otsu's method estimates an intensity threshold separating two groups of pixels. Choose whether your objects are brighter or darker than the background.

In [ ]:
# Change this to False if candidate cells are darker than the background.
bright_cells = True

# Smooth the image and calculate an automatic intensity threshold.
smoothed = filters.gaussian(gray, sigma=1)
threshold = filters.threshold_otsu(smoothed)
print("Otsu threshold:", threshold)

# Create a Boolean mask: True represents candidate foreground pixels.
if bright_cells:
    mask = smoothed > threshold
else:
    mask = smoothed < threshold

plt.imshow(mask, cmap="gray")
plt.title("Initial foreground mask")
plt.axis("off")
plt.show()

### 6.3 Clean the mask and label connected regions
Small foreground objects can be noise; small holes can occur inside candidate cells. The size thresholds below are in pixels of the retrieved representation, not physical units.

In [ ]:
# Remove small foreground objects and fill small holes.
minimum_area = 30
mask_clean = morphology.remove_small_objects(mask, min_size=minimum_area)
mask_clean = morphology.remove_small_holes(mask_clean, area_threshold=30)

# Assign a unique integer label to each connected foreground region.
labels = measure.label(mask_clean)

# Calculate properties such as area, bounding box and centroid.
regions = measure.regionprops(labels)
print("Number of connected regions:", len(regions))

In [ ]:
# Compare the input image, cleaned mask and coloured region overlay.
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(analysis_image)
axes[0].set_title("Input")
axes[1].imshow(mask_clean, cmap="gray")
axes[1].set_title("Candidate foreground")
axes[2].imshow(color.label2rgb(labels, image=gray, bg_label=0, alpha=0.35))
axes[2].set_title(f"{len(regions)} connected regions")

# Hide the coordinate axes on each panel.
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 6.4 Highlight detected regions on the image
For each connected region, draw its bounding box and centroid. NumPy uses row, column coordinates (y, x), whereas Matplotlib plots x, y.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(analysis_image)

for region in regions:
    # Extract the bounding box and centre in array coordinates.
    y0, x0, y1, x1 = region.bbox
    cy, cx = region.centroid

    # Draw a rectangle around the region using plotting coordinates.
    box = Rectangle((x0, y0), x1 - x0, y1 - y0,
                    fill=False, edgecolor="cyan", linewidth=1)
    ax.add_patch(box)

    # Mark the centre of the region with a yellow cross.
    ax.plot(cx, cy, "+", color="yellow")

ax.set_title("Candidate cell regions — inspect visually")
ax.axis("off")
plt.show()

## 7. Interpret your results
Try changing `bright_cells`, `sigma` in the Gaussian filter and `minimum_area`. Rerun the affected cells and all subsequent analysis cells.

- Which cells are missed? Which objects are merged?
- How do detections compare with your manual annotations, if you chose the same image and region?
- Why are connected regions not necessarily individual cells?

Touching cells, debris and uneven illumination can bias counts. Physical size and viability cannot be inferred without calibration and experimental evidence.

## 8. Wrap-up
What can we share and reuse? The original image service, the enriched manifest and the analysis code. Translating computational detections into IIIF annotations would require accounting for crop offsets, resizing and the mapping between source-image and Canvas coordinates.

### References
- [IIIF Presentation API 3](https://iiif.io/api/presentation/3.0/)
- [IIIF Image API 3](https://iiif.io/api/image/3.0/)
- [4TU API documentation](https://data.4tu.nl/api/docs?urls.primaryName=v2)
- [Digirati annotation documentation](https://manifest-editor-docs.netlify.app/docs/creating-annotations)

### Instructor preflight
The original notebook reported that the sandbox endpoints could not be retrieved during preparation. This revision has not been live-tested against the dataset. Verify access from Colab, confirm the file-list schema and selected image UUID, check supported sizes and cropping, and tune segmentation parameters before teaching. If arbitrary resizing is unsupported, use an advertised size (or skip the optional crop).